# ONG v3 - Pilot: DINOv2 (corrected, prior-respecting recipe)

**Run top-to-bottom (Runtime -> Run all).** Trains ONE model - DINOv2 ViT-L/14 - with the
corrected recipe `--sampler-power 0 --loss ce` (no double-balancing; respects the true
Bulbophyllum/Dendrobium prior). Saves BOTH `best_model_macro.pth` and
`best_model_global.pth`, evaluates BOTH on the frozen test split, and prints a comparison
table vs the baseline.

**Baseline to beat** (FROZEN deterministic split, 16,701 img / 120 genera; OPTIMISTIC -
train/test overlap, so even matching it is a win): macro top-1 **74.0%** (>=5: 76.2%),
global 90.1%, species R@5 73.4%, genus R@5 94.1%.

**Before running**, upload to Drive `MyDrive/ONG_v3/`:
`scripts/03_train_bakeoff_colab.py`, `scripts/13_evaluate.py`, `data/splits/*.csv`,
`photos.zip`. About 15-20 compute units on an L4. (The full 4-model bake-off later =
`scripts/run_bakeoff_all_colab.py`, run separately - not needed for this pilot.)

In [ ]:
# Sel 1 - Mount Drive, install deps, check GPU
from google.colab import drive; drive.mount('/content/drive')
!pip -q install -U timm open_clip_torch faiss-cpu
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert torch.cuda.is_available(), 'No GPU - Runtime > Change runtime type > L4, then Run all again.'

In [ ]:
# Sel 2 - Set ROOT and verify required files are on Drive
import os
ROOT = '/content/drive/MyDrive/ONG_v3'
need = ['scripts/03_train_bakeoff_colab.py', 'scripts/13_evaluate.py',
        'data/splits/train_live.csv', 'photos.zip']
status = {f: os.path.exists(f'{ROOT}/{f}') for f in need}
print('present:', status)
missing = [f for f, ok in status.items() if not ok]
assert not missing, f'Missing on Drive under {ROOT}/: {missing} - upload them, then Run all again.'
print('All required files present.')

In [ ]:
# Sel 3 - Unzip photos to fast local disk (/content/photos/{Genus}/*.jpg)
!unzip -q -o "{ROOT}/photos.zip" -d /content/
assert os.path.isdir('/content/photos'), 'Expected /content/photos after unzip - check the zip layout.'
print('genera folders:', len(os.listdir('/content/photos')))

In [ ]:
# Sel 4 - Train DINOv2 (corrected recipe) + evaluate BOTH selection checkpoints
MODEL, EVAL_NAME, IMG = 'dinov2l', 'vit_large_patch14_reg4_dinov2.lvd142m', 448

# Train: no double-balancing (respect the prior). Saves best_model_macro.pth + best_model_global.pth.
!python {ROOT}/scripts/03_train_bakeoff_colab.py --model {MODEL} --drive-root {ROOT} --photos-root /content/photos --sampler-power 0 --loss ce

# Evaluate each checkpoint on the frozen test split:
for tag in ['macro', 'global']:
    ckpt = f'{ROOT}/models/{MODEL}/best_model_{tag}.pth'
    out  = f'{ROOT}/eval/{MODEL}' if tag == 'macro' else f'{ROOT}/eval/{MODEL}_{tag}'
    assert os.path.exists(ckpt), f'Checkpoint missing: {ckpt} - training likely failed (see log above).'
    print(f'\n{"="*70}\n[EVAL] {MODEL} [{tag}-selected]\n{"="*70}')
    get_ipython().system(
        f'python {ROOT}/scripts/13_evaluate.py --model {EVAL_NAME} --img-size {IMG} '
        f'--checkpoint {ckpt} --vocab {ROOT}/models/{MODEL}/vocab.json '
        f'--splits-dir {ROOT}/data/splits --photos-root /content/photos --out-dir {out}'
    )

In [ ]:
# Sel 5 - Comparison table (reads every eval/<dir>/results.json) + save markdown summary
import json, datetime
from pathlib import Path

BASE = dict(name='baseline effb4 (FROZEN)', macro_top1=74.0, macro_top5=84.8,
            global_top1=90.1, sp_r5=73.4, gn_r5=94.1, ece=0.082)
PRETTY = {'bioclip2': 'BioCLIP-2 ViT-L/14', 'dinov2l': 'DINOv2 ViT-L/14',
          'convnextv2l': 'ConvNeXt-V2-L', 'effnetv2l': 'EfficientNetV2-L'}

def pretty(name):
    suf = ''
    if name.endswith('_global'):
        name, suf = name[:-7], ' [global-sel]'
    return PRETTY.get(name, name) + suf

def read(p):
    r = json.loads(Path(p).read_text())
    c = r.get('classification', {}) or {}
    q = r.get('retrieval', {}) or {}
    pc = lambda d, k: round(d[k] * 100, 1) if d.get(k) is not None else None
    return dict(macro_top1=pc(c, 'macro_top1'), macro_top5=pc(c, 'macro_top5'),
                global_top1=pc(c, 'global_top1'), sp_r5=pc(q, 'species_recall@5'),
                gn_r5=pc(q, 'genus_recall@5'),
                ece=(round(c['ece'], 3) if c.get('ece') is not None else None))

evdir = Path(ROOT) / 'eval'
found = sorted(p.parent.name for p in evdir.glob('*/results.json')
               if not p.parent.name.startswith('_'))
f2 = lambda x: '-' if x is None else f'{x}'
date = datetime.date.today().isoformat()
L = [f'## {date} - Pilot results (DINOv2 corrected recipe; frozen-split protocol)',
     '',
     'Baseline = FROZEN deterministic effb4 (OPTIMISTIC: train/test overlap; macro>=5 76.2%). '
     '`[global-sel]` = checkpoint chosen by val global top-1.',
     '',
     '| Model | macro T1 | macro T5 | global T1 | species R@5 | genus R@5 | ECE |',
     '|-------|----------|----------|-----------|-------------|-----------|-----|',
     f"| {BASE['name']} | **{BASE['macro_top1']}** | {BASE['macro_top5']} | {BASE['global_top1']} | {BASE['sp_r5']} | {BASE['gn_r5']} | {BASE['ece']} |"]
best_cls = best_ret = None
for name in found:
    r = read(evdir / name / 'results.json')
    L.append(f"| {pretty(name)} | **{f2(r['macro_top1'])}** | {f2(r['macro_top5'])} | "
             f"{f2(r['global_top1'])} | {f2(r['sp_r5'])} | {f2(r['gn_r5'])} | {f2(r['ece'])} |")
    if r['macro_top1'] is not None and (best_cls is None or r['macro_top1'] > best_cls[1]):
        best_cls = (name, r['macro_top1'])
    if r['sp_r5'] is not None and (best_ret is None or r['sp_r5'] > best_ret[1]):
        best_ret = (name, r['sp_r5'])
L.append('')
if best_cls:
    d = round(best_cls[1] - BASE['macro_top1'], 1)
    L.append(f"- **Best genus (macro top-1):** {pretty(best_cls[0])} = {best_cls[1]}% "
             f"({'+' if d >= 0 else ''}{d} vs baseline).")
if best_ret:
    d = round(best_ret[1] - BASE['sp_r5'], 1)
    L.append(f"- **Best retrieval (species R@5):** {pretty(best_ret[0])} = {best_ret[1]}% "
             f"({'+' if d >= 0 else ''}{d} vs baseline).")
snippet = '\n'.join(L)
out = evdir / f'pilot_summary_{date}.md'
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(snippet, encoding='utf-8')
print(snippet)
print(f'\nSaved -> {out}  (download via the Files panel; paste into ONGv3_progress.Rmd)')